# CUTEst

In [ ]:
import os

from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.util.method import get_methods
from qnlab.experiment.for_cutest_run import run, load_results
from qnlab.experiment.for_cutest_vis import draw_pp, individual_plot

In [ ]:
os.chdir(Path(os.path.abspath("cutest.ipynb")).parent.parent.resolve())
print(os.getcwd())

In [ ]:
def main():
    ERROR_CAUSING_TASKS = [
        (16, "INDEFM", "SciPy"),
    ]

    for precision, noise in [
        # (64, np.float64(0.0)),
        # (32, np.float64(0.0)),
        # (16, np.float64(0.0)),
        # (64, np.float64(1e-3)),
    ]:
        problems = problemsToRun(precision)
        methods, *_ = get_methods()
        if noise > 0:
            new_methods = []
            for method, option in methods:
                new_option = option.copy()
                new_option["gtol"] = noise * 10
                new_methods.append((method, new_option))
            methods = new_methods
        run(problems, methods, precision, noise, ERROR_CAUSING_TASKS, TL=600)

    methods, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_methods()
    for precision, _noise, _gtol in [
        (64, 0.0, 1e-1),
        (64, 0.0, 1e-3),
        (64, 0.0, 1e-5),
        (32, 0.0, 1e-1),
        (32, 0.0, 1e-3),
        (32, 0.0, 1e-5),
        (16, 0.0, 1e-1),
        (16, 0.0, 1e-3),
        (16, 0.0, 1e-5),
        (64, 1e-3, 1e-2),
    ]:
        noise = np.float64(_noise)
        gtol = np.float64(_gtol)
        problems = problemsToRun(precision)

        alg_names, callsM, fxsM, gnormsM, problems = load_results(
            methods, problems, precision, noise, gtol
        )
        draw_pp(
            alg_names,
            callsM,
            ALGORITHM_COLORS,
            ALGORITHM_LINE_STYLES,
            precision,
            noise,
            gtol,
        )

        if False:
            individual_plot(problems, methods, precision, noise)
        if True:
            data = {}
            data["problem"] = problems
            for i, alg_name in enumerate(alg_names):
                data[f"{alg_name}"] = callsM[i, :].tolist()
            df = pd.DataFrame(data)
            df.set_index("problem", inplace=True)

            # Apply styling
            def color_scale_with_cmap(row):
                if np.all(np.isinf(row.values)):
                    return ["background-color: rgba(0, 0, 0, 0.8)" for _ in row.values]
                norm = plt.Normalize(vmin=row.min(), vmax=row.min() * 10)  # type:ignore
                cmap = matplotlib.colormaps["coolwarm"]
                return [
                    f"background-color: rgba({int(r * 255)}, {int(g * 255)}, {int(b * 255)}, 0.8)"
                    for r, g, b, _ in cmap(norm(row.values))
                ]

            styled_df = df.style.apply(color_scale_with_cmap, axis=1)
            display(styled_df)


In [ ]:
main()